# 🌊 AquaSentinel AI: Production Training & Evaluation Pipeline
### Multi-Phase Side-Scan Sonar Segmentation (GhostVision + SSS-Mine / NOMBO)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RaghavKacker/Aqua-Sentinel/blob/main/notebooks/AquaSentinel_Training.ipynb)

This notebook follows the 11-step project specification to inspect real datasets, normalize annotations, train a YOLO-Seg segmentation model on Google Colab GPU, and evaluate performance with acoustic shadow verification.

```text
1. Mount Google Drive
2. Install dependencies
3. Locate / Download GhostVision
4. Locate / Download SSS-Mine
5. Inspect both datasets          ← CURRENT STEP
6. Convert annotations (YOLO-Seg)
7. Create train / val / test (Survey-Isolated)
8. Train YOLO-Seg
9. Validate
10. Test
11. Compare Results (YOLO vs YOLO + Acoustic Gate)
```

---

## Phase 1: Mount Google Drive
Mount your Google Drive to access or store your dataset archives and model checkpoints.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully!")

## Phase 2: Install Dependencies
Installs Ultralytics, OpenCV, and tools for hydrographic dataset parsing.

In [ ]:
!nvidia-smi
!pip install -q ultralytics opencv-python-headless pyyaml matplotlib tabulate

import torch, ultralytics
print(f"\nPyTorch Version: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
ultralytics.checks()

## Phase 3 & 4: Locate Downloaded Datasets
Locates where **GhostVision** and **SSS-Mine** were downloaded or unzipped in your Colab session or Google Drive.

In [ ]:
import os, glob
from pathlib import Path

# Search for dataset folders in /content and /content/drive
def find_dataset_dir(candidates):
    for c in candidates:
        if os.path.exists(c):
            return Path(c)
    return None

ghostvision_candidates = [
    './GhostVision', './data/GhostVision', '/content/GhostVision',
    '/content/drive/MyDrive/GhostVision', '/content/drive/MyDrive/AquaSentinel_Data/GhostVision'
]
sssmine_candidates = [
    './SSS-Mine', './NOMBO', './data/SSS-Mine', '/content/SSS-Mine',
    '/content/drive/MyDrive/SSS-Mine', '/content/drive/MyDrive/AquaSentinel_Data/SSS-Mine'
]

GHOSTVISION_PATH = find_dataset_dir(ghostvision_candidates)
SSSMINE_PATH = find_dataset_dir(sssmine_candidates)

print(f"GhostVision Path: {GHOSTVISION_PATH if GHOSTVISION_PATH else 'NOT FOUND (Check path)'}")
print(f"SSS-Mine Path:    {SSSMINE_PATH if SSSMINE_PATH else 'NOT FOUND (Check path)'}")

# If your path is different, set it manually below:
# GHOSTVISION_PATH = Path('/content/your_ghostvision_folder')
# SSSMINE_PATH = Path('/content/your_sssmine_folder')

## Phase 5: Deep Inspection of Both Datasets (CURRENT STEP)
Examines directory structures, file counts, image formats, annotation types (COCO, YOLO, Pascal VOC), and class names.

In [ ]:
import json, glob, xml.etree.ElementTree as ET
import cv2, numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from tabulate import tabulate

def audit_dataset(dataset_name, root_path):
    print("=" * 70)
    print(f"AUDIT REPORT: {dataset_name.upper()}")
    print("=" * 70)
    if not root_path or not os.path.exists(root_path):
        print(f"[ERROR] Path does not exist: {root_path}")
        return {}

    root = Path(root_path)
    img_exts = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tif', '*.pbm']
    all_images = []
    for ext in img_exts:
        all_images.extend(glob.glob(str(root / '**' / ext), recursive=True))

    json_files = glob.glob(str(root / '**' / '*.json'), recursive=True)
    xml_files = glob.glob(str(root / '**' / '*.xml'), recursive=True)
    txt_files = glob.glob(str(root / '**' / '*.txt'), recursive=True)
    csv_files = glob.glob(str(root / '**' / '*.csv'), recursive=True)

    print(f"Root Path:         {root.resolve()}")
    print(f"Total Images:      {len(all_images)}")
    print(f"JSON Files:        {len(json_files)}")
    print(f"XML Files:         {len(xml_files)}")
    print(f"TXT Files:         {len(txt_files)}")
    print(f"CSV Logs:          {len(csv_files)}")

    # Sample image resolutions
    resolutions = set()
    for img_p in all_images[:20]:
        im = cv2.imread(img_p)
        if im is not None:
            resolutions.add(f"{im.shape[1]}x{im.shape[0]} (Channels: {im.shape[2] if len(im.shape) == 3 else 1})")
    print(f"Sample Dimensions: {list(resolutions)[:3]}")

    # Annotation inspection
    classes_found = Counter()
    anno_type = "Unknown"

    if json_files:
        anno_type = "COCO / Custom JSON"
        for jf in json_files[:3]:
            try:
                with open(jf, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    if 'categories' in data:
                        for cat in data['categories']:
                            classes_found[cat.get('name', cat.get('id'))] += 1
            except Exception:
                pass

    if xml_files:
        anno_type = "Pascal VOC XML"
        for xf in xml_files[:100]:
            try:
                tree = ET.parse(xf)
                for obj in tree.findall('object'):
                    name = obj.find('name').text
                    classes_found[name] += 1
            except Exception:
                pass

    if txt_files and not classes_found:
        anno_type = "YOLO / Text Format"
        for tf in txt_files[:200]:
            try:
                with open(tf, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if parts:
                            classes_found[f"class_{parts[0]}"] += 1
            except Exception:
                pass

    print(f"Detected Annotation Format: {anno_type}")
    print(f"Unique Classes Discovered:  {dict(classes_found)}\n")
    return {
        'images': all_images,
        'anno_type': anno_type,
        'classes': classes_found,
        'json_files': json_files,
        'txt_files': txt_files
    }

# Run inspection
gv_info = audit_dataset("GhostVision", GHOSTVISION_PATH)
mine_info = audit_dataset("SSS-Mine / NOMBO", SSSMINE_PATH)

### Visual Gallery: Inspect Raw Sonar Images
Displays a row of raw images from both datasets to inspect acoustic shadows, highlights, and seabed reverberation.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
sample_gv = gv_info.get('images', [])[:2]
sample_mine = mine_info.get('images', [])[:2]
all_samples = sample_gv + sample_mine

if all_samples:
    for i, p in enumerate(all_samples):
        im = cv2.imread(p)
        if im is not None:
            im_rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
            axes[i].imshow(im_rgb)
            axes[i].set_title(f"{Path(p).parent.name} / {Path(p).name}", fontsize=8)
            axes[i].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No sample images found to display. Please verify dataset paths.")

## Phase 6: Convert Annotations (Planned Next Step)
Converts the inspected labels into normalized Ultralytics YOLO-Seg polygon segmentation format.
Once Phase 5 inspection prints your exact class names, run this conversion cell.

In [ ]:
print("Ready for Phase 6 (Convert Annotations).")
print("After checking the classes above, this cell maps raw names to canonical classes:")
print("0: crab_pot | 1: ghost_gear | 2: mine_cylinder | 3: debris_anomaly")

## Phase 7: Create Survey-Isolated Train / Val / Test Splits
Groups tiles by survey mission (not random image splitting) to prevent ping leakage across splits.

In [ ]:
print("Ready for Phase 7 (Survey/Mission Splitting & dataset.yaml generation).")

## Phase 8: Train YOLO-Seg Model on Colab GPU

In [ ]:
# Pre-configured training command
# model = YOLO('yolo11n-seg.pt')
# results = model.train(data='./data/unified_yolo_seg/dataset.yaml', epochs=35, imgsz=640, batch=16, device=0)
print("Ready for Phase 8 (Model Training).")

## Phase 9 & 10: Validate & Test on Unseen Mission

In [ ]:
# metrics = model.val(data='./data/unified_yolo_seg/dataset.yaml', split='test')
print("Ready for Phase 9 & 10 (Validation & Unseen Test Evaluation).")

## Phase 11: Compare Results (YOLO-Only vs. YOLO + Acoustic Verification)

In [ ]:
print("Ready for Phase 11 (Comparative Benchmark on False Positives).")